In [1]:
%%writefile Algorithm_4_in_cpp_parallel_computing_variant5.cpp

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdlib>
#include <functional>
#include <iomanip>
#include <iostream>
#include <limits>
#include <memory>
#include <numeric>
#include <queue>
#include <random>
#include <stdexcept>
#include <string>
#include <thread>
#include <tuple>
#include <utility>
#include <vector>


// Pipeline:
//   1. Compute an MST of the dense input graph.
//   2. Build a Kruskal reconstruction tree from the MST.
//   3. Put the original vertices in reconstruction-tree leaf order.  Every
//      reconstruction-tree subtree is then a contiguous leaf interval.
//   4. An internal reconstruction-tree node with children A and B says that
//      every pair in A x B has minimax distance equal to that node's weight.
//      Fill those two rectangular matrix regions.
//   5. Threads own disjoint output-row ranges.  Consequently no output lock,
//      atomic operation, or concurrent write to the same cache line is needed
//      in the main filling phase.
//   6. Optionally restore the matrix to the original vertex numbering.

namespace mmj {

using Clock = std::chrono::steady_clock;

double seconds_between(Clock::time_point a, Clock::time_point b) {
    return std::chrono::duration<double>(b - a).count();
}

class DenseMatrix {
public:
    DenseMatrix() = default;

    // Memory is intentionally uninitialized unless zero_initialize is true.
    // The MMJ filling algorithm writes every off-diagonal entry exactly once
    // and explicitly writes every diagonal entry.
    explicit DenseMatrix(std::size_t n, bool zero_initialize = false)
        : n_(n) {
        if (n_ != 0 && n_ > std::numeric_limits<std::size_t>::max() / n_) {
            throw std::overflow_error("matrix element count overflow");
        }
        const std::size_t count = n_ * n_;
        if (count != 0) {
            data_ = std::unique_ptr<double[]>(new double[count]);
            if (zero_initialize) {
                std::fill_n(data_.get(), count, 0.0);
            }
        }
    }

    DenseMatrix(DenseMatrix&&) noexcept = default;
    DenseMatrix& operator=(DenseMatrix&&) noexcept = default;
    DenseMatrix(const DenseMatrix&) = delete;
    DenseMatrix& operator=(const DenseMatrix&) = delete;

    std::size_t n() const noexcept { return n_; }
    std::size_t elements() const noexcept { return n_ * n_; }

    double* row(std::size_t i) noexcept { return data_.get() + i * n_; }
    const double* row(std::size_t i) const noexcept {
        return data_.get() + i * n_;
    }

    double& operator()(std::size_t i, std::size_t j) noexcept {
        return data_[i * n_ + j];
    }
    const double& operator()(std::size_t i, std::size_t j) const noexcept {
        return data_[i * n_ + j];
    }

    void release() noexcept {
        data_.reset();
        n_ = 0;
    }

private:
    std::size_t n_ = 0;
    std::unique_ptr<double[]> data_;
};

struct Edge {
    int u;
    int v;
    double weight;
};

// Heap-based Prim for a dense matrix.  It scans one dense row for each chosen
// vertex, while the heap avoids a second O(n^2) scan for the next vertex.
std::vector<Edge> prim_mst_dense(const DenseMatrix& distance) {
    const std::size_t n_size = distance.n();
    if (n_size == 0) {
        return {};
    }
    if (n_size > static_cast<std::size_t>(std::numeric_limits<int>::max())) {
        throw std::invalid_argument("too many vertices for 32-bit indices");
    }
    const int n = static_cast<int>(n_size);
    const double inf = std::numeric_limits<double>::infinity();

    std::vector<double> key(n, inf);
    std::vector<int> parent(n, -1);
    std::vector<std::uint8_t> in_mst(n, 0);

    using HeapItem = std::pair<double, int>;
    std::priority_queue<HeapItem, std::vector<HeapItem>,
                        std::greater<HeapItem>> heap;
    key[0] = 0.0;
    heap.emplace(0.0, 0);

    int chosen = 0;
    while (!heap.empty()) {
        const auto [ignored_key, u] = heap.top();
        (void)ignored_key;
        heap.pop();
        if (in_mst[u]) {
            continue;
        }
        in_mst[u] = 1;
        ++chosen;

        const double* drow = distance.row(static_cast<std::size_t>(u));
        for (int v = 0; v < n; ++v) {
            const double w = drow[v];
            if (!in_mst[v] && w < key[v]) {
                key[v] = w;
                parent[v] = u;
                heap.emplace(w, v);
            }
        }
    }

    if (chosen != n) {
        throw std::runtime_error("input graph is disconnected");
    }

    std::vector<Edge> mst;
    mst.reserve(n > 0 ? static_cast<std::size_t>(n - 1) : 0);
    for (int v = 1; v < n; ++v) {
        if (parent[v] < 0) {
            throw std::runtime_error("Prim failed to assign a parent");
        }
        mst.push_back({parent[v], v, distance(v, parent[v])});
    }
    return mst;
}

class DisjointSet {
public:
    explicit DisjointSet(int n)
        : parent_(n), size_(n, 1), tree_root_(n) {
        std::iota(parent_.begin(), parent_.end(), 0);
        std::iota(tree_root_.begin(), tree_root_.end(), 0);
    }

    int find(int x) {
        int root = x;
        while (parent_[root] != root) {
            root = parent_[root];
        }
        while (parent_[x] != x) {
            const int next = parent_[x];
            parent_[x] = root;
            x = next;
        }
        return root;
    }

    int tree_root_of_set(int set_root) const { return tree_root_[set_root]; }

    int unite_roots(int a, int b, int new_tree_root) {
        if (size_[a] < size_[b]) {
            std::swap(a, b);
        }
        parent_[b] = a;
        size_[a] += size_[b];
        tree_root_[a] = new_tree_root;
        return a;
    }

private:
    std::vector<int> parent_;
    std::vector<int> size_;
    std::vector<int> tree_root_;
};

struct ReconstructionNode {
    int left = -1;
    int right = -1;
    double weight = 0.0;
    int begin = -1;  // inclusive leaf-order position
    int end = -1;    // exclusive leaf-order position
};

struct ReconstructionTree {
    int original_vertex_count = 0;
    int root = -1;
    std::vector<ReconstructionNode> nodes;
    // leaf_order[position] = original vertex id
    std::vector<int> leaf_order;
    // position[original vertex id] = position in leaf_order
    std::vector<int> position;
};

ReconstructionTree build_reconstruction_tree(
    int n, std::vector<Edge> mst_edges) {
    if (n <= 0) {
        return {};
    }
    if (static_cast<int>(mst_edges.size()) != n - 1) {
        throw std::invalid_argument("an n-vertex MST must contain n-1 edges");
    }

    std::sort(mst_edges.begin(), mst_edges.end(),
              [](const Edge& a, const Edge& b) {
                  if (a.weight != b.weight) return a.weight < b.weight;
                  if (a.u != b.u) return a.u < b.u;
                  return a.v < b.v;
              });

    ReconstructionTree tree;
    tree.original_vertex_count = n;
    tree.nodes.resize(static_cast<std::size_t>(2 * n - 1));
    tree.leaf_order.resize(n);
    tree.position.resize(n);

    DisjointSet dsu(n);
    int next_node = n;

    for (const Edge& edge : mst_edges) {
        int a = dsu.find(edge.u);
        int b = dsu.find(edge.v);
        if (a == b) {
            throw std::runtime_error("input edge set is not a tree");
        }

        const int left_root = dsu.tree_root_of_set(a);
        const int right_root = dsu.tree_root_of_set(b);
        ReconstructionNode& node = tree.nodes[next_node];
        node.left = left_root;
        node.right = right_root;
        node.weight = edge.weight;

        dsu.unite_roots(a, b, next_node);
        ++next_node;
    }

    tree.root = dsu.tree_root_of_set(dsu.find(0));

    // Iterative postorder traversal: safe even for a completely skewed tree.
    std::vector<std::pair<int, bool>> stack;
    stack.reserve(static_cast<std::size_t>(4 * n));
    stack.emplace_back(tree.root, false);
    int cursor = 0;

    while (!stack.empty()) {
        const auto [node_id, expanded] = stack.back();
        stack.pop_back();
        ReconstructionNode& node = tree.nodes[node_id];

        if (node_id < n) {
            node.begin = cursor;
            node.end = cursor + 1;
            tree.leaf_order[cursor] = node_id;
            tree.position[node_id] = cursor;
            ++cursor;
            continue;
        }

        if (!expanded) {
            stack.emplace_back(node_id, true);
            // Push right first so that left is processed first.
            stack.emplace_back(node.right, false);
            stack.emplace_back(node.left, false);
        } else {
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];
            node.begin = left.begin;
            node.end = right.end;
            if (left.end != right.begin) {
                throw std::runtime_error("non-contiguous reconstruction subtree");
            }
        }
    }

    if (cursor != n) {
        throw std::runtime_error("reconstruction tree does not contain all leaves");
    }
    return tree;
}

unsigned normalized_thread_count(unsigned requested, std::size_t n) {
    unsigned threads = requested;
    if (threads == 0) {
        threads = std::thread::hardware_concurrency();
    }
    if (threads == 0) {
        threads = 1;
    }
    if (n != 0) {
        threads = std::min<unsigned>(threads, static_cast<unsigned>(n));
    }
    return std::max(1u, threads);
}

template <class Function>
void parallel_rows(std::size_t n, unsigned requested_threads, Function fn) {
    const unsigned threads = normalized_thread_count(requested_threads, n);
    std::vector<std::thread> workers;
    workers.reserve(threads > 0 ? threads - 1 : 0);

    auto run = [&](unsigned tid) {
        const std::size_t begin = n * tid / threads;
        const std::size_t end = n * (tid + 1) / threads;
        fn(begin, end);
    };

    // The caller thread owns the last partition.  Total active computation
    // threads therefore equals 'threads', not threads+1.
    for (unsigned tid = 0; tid + 1 < threads; ++tid) {
        workers.emplace_back(run, tid);
    }
    run(threads - 1);
    for (std::thread& worker : workers) {
        worker.join();
    }
}

DenseMatrix fill_in_leaf_order(const ReconstructionTree& tree,
                               unsigned requested_threads) {
    const int n = tree.original_vertex_count;
    DenseMatrix output(static_cast<std::size_t>(n), false);

    parallel_rows(static_cast<std::size_t>(n), requested_threads,
                  [&](std::size_t owned_begin, std::size_t owned_end) {
        // Every thread owns complete rows.  It scans the small O(n) list of
        // internal nodes and writes only intersections with its row range.
        for (int node_id = n; node_id < 2 * n - 1; ++node_id) {
            const ReconstructionNode& node = tree.nodes[node_id];
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];

            const std::size_t left_begin = static_cast<std::size_t>(left.begin);
            const std::size_t left_end = static_cast<std::size_t>(left.end);
            const std::size_t right_begin = static_cast<std::size_t>(right.begin);
            const std::size_t right_end = static_cast<std::size_t>(right.end);

            // Owned rows in the left child: fill their right-child interval.
            const std::size_t lr_begin = std::max(owned_begin, left_begin);
            const std::size_t lr_end = std::min(owned_end, left_end);
            for (std::size_t row = lr_begin; row < lr_end; ++row) {
                std::fill(output.row(row) + right_begin,
                          output.row(row) + right_end, node.weight);
            }

            // Owned rows in the right child: fill their left-child interval.
            const std::size_t rr_begin = std::max(owned_begin, right_begin);
            const std::size_t rr_end = std::min(owned_end, right_end);
            for (std::size_t row = rr_begin; row < rr_end; ++row) {
                std::fill(output.row(row) + left_begin,
                          output.row(row) + left_end, node.weight);
            }
        }

        for (std::size_t row = owned_begin; row < owned_end; ++row) {
            output(row, row) = 0.0;
        }
    });
    return output;
}

// Convert P[leaf_position(i), leaf_position(j)] into M[i, j].
// Each source and destination row stays private to one worker.  Within a row,
// the source is read sequentially; the destination permutation usually fits
// in the core's private cache for the graph sizes targeted here.
DenseMatrix restore_original_order(const DenseMatrix& permuted,
                                   const ReconstructionTree& tree,
                                   unsigned requested_threads) {
    const std::size_t n = permuted.n();
    DenseMatrix restored(n, false);

    parallel_rows(n, requested_threads,
                  [&](std::size_t original_begin, std::size_t original_end) {
        for (std::size_t original_row = original_begin;
             original_row < original_end; ++original_row) {
            const std::size_t source_position = static_cast<std::size_t>(
                tree.position[original_row]);
            const double* source = permuted.row(source_position);
            double* destination = restored.row(original_row);

            for (std::size_t leaf_position = 0; leaf_position < n;
                 ++leaf_position) {
                const std::size_t original_column = static_cast<std::size_t>(
                    tree.leaf_order[leaf_position]);
                destination[original_column] = source[leaf_position];
            }
        }
    });
    return restored;
}

struct Result {
    DenseMatrix matrix;
    ReconstructionTree reconstruction_tree;
    bool in_original_order = false;
};

Result compute_from_mst(int n, const std::vector<Edge>& mst,
                        unsigned requested_threads,
                        bool restore_vertex_order) {
    ReconstructionTree tree = build_reconstruction_tree(n, mst);
    DenseMatrix permuted = fill_in_leaf_order(tree, requested_threads);

    DenseMatrix final_matrix;
    if (restore_vertex_order) {
        final_matrix = restore_original_order(permuted, tree, requested_threads);
        permuted.release();
    } else {
        final_matrix = std::move(permuted);
    }

    return {std::move(final_matrix), std::move(tree), restore_vertex_order};
}

DenseMatrix create_symmetric_distance_matrix(int n, std::uint32_t seed) {
    if (n < 0) {
        throw std::invalid_argument("n must be nonnegative");
    }
    DenseMatrix matrix(static_cast<std::size_t>(n), false);
    std::mt19937 generator(seed);
    std::uniform_real_distribution<double> distribution(1.0, 19999.0);

    for (int i = 0; i < n; ++i) {
        matrix(i, i) = 0.0;
        for (int j = i + 1; j < n; ++j) {
            const double value = std::round(distribution(generator) * 100.0) /
                                 100.0;
            matrix(i, j) = value;
            matrix(j, i) = value;
        }
    }
    return matrix;
}

double value_in_original_numbering(const Result& result, int i, int j) {
    if (result.in_original_order) {
        return result.matrix(i, j);
    }
    const int pi = result.reconstruction_tree.position[i];
    const int pj = result.reconstruction_tree.position[j];
    return result.matrix(pi, pj);
}

}  // namespace mmj

int main() {
    // WARNING:
    // Dense matrices explode in memory quickly.
 

    const int N = 10000;

    unsigned n_jobs = std::thread::hardware_concurrency();
    if (n_jobs == 0) {
        n_jobs = 1;
    }

    const std::uint32_t random_seed = 222;
    const bool restore_original_order = true;

    std::cout << "Number of nodes: "
              << N << '\n';

    std::cout << "Number of CPU cores: "
              << mmj::normalized_thread_count(n_jobs, N) << '\n';

    auto distance_matrix =
        mmj::create_symmetric_distance_matrix(
            N,
            random_seed
        );

    const auto start = mmj::Clock::now();

    auto mst =
        mmj::prim_mst_dense(
            distance_matrix
        );

    distance_matrix.release();

    auto result =
        mmj::compute_from_mst(
            N,
            mst,
            n_jobs,
            restore_original_order
        );

    const auto end = mmj::Clock::now();

    const double time_used =
        mmj::seconds_between(
            start,
            end
        );

    std::cout << std::fixed
              << std::setprecision(3);

    std::cout << "Time used for MMJ matrix "
                 "(Variant 5 of Algorithm 13): "
              << time_used
              << " seconds\n";

    std::cout << "Print last 30 values of first row "
                 "of MMJ matrix:\n";

    const int first = std::max(0, N - 30);

    for (int j = first; j < N; ++j) {
        std::cout << std::fixed
                  << std::setprecision(2)
                  << mmj::value_in_original_numbering(
                         result,
                         0,
                         j
                     )
                  << " ";
    }

    std::cout << "\n";

    return 0;
}

Overwriting Algorithm_4_in_cpp_parallel_computing_variant5.cpp


In [2]:
!g++ -std=c++17 -O3 -march=native -flto \
    -DNDEBUG -pthread \
    Algorithm_4_in_cpp_parallel_computing_variant5.cpp \
    -o Algorithm_4_in_cpp_parallel_computing_variant5

In [3]:
!./Algorithm_4_in_cpp_parallel_computing_variant5

Number of nodes: 10000
Number of CPU cores: 4
Time used for MMJ matrix (Variant 5 of Algorithm 13): 0.779 seconds
Print last 30 values of first row of MMJ matrix:
4.87 5.61 3.88 3.43 7.33 5.15 3.55 3.26 3.48 3.26 3.38 3.26 5.52 5.03 6.59 3.49 5.03 3.26 5.75 3.98 3.26 4.58 5.19 3.97 3.26 3.45 3.30 3.44 10.70 3.42 
